# Building an AI agent — and turning what it does into training material

This shows a small, real, free open-source AI model acting as an assistant that can answer questions, and — when it doesn't already know something — look it up using a simple tool, the way a person might search something up rather than guess. Everything below runs live: a real AI model, doing real work, on a real graphics processor (GPU). Nothing is pre-recorded or faked.

The second half shows *why this matters commercially*: everything the assistant does while answering is automatically captured in a clean, structured format that can later be used to train and improve the assistant — the same record-keeping a company would use to make its AI get better over time from its own real usage.


In [1]:
import os
os.environ.setdefault("HF_HUB_OFFLINE", "1")
os.environ.setdefault("TRANSFORMERS_OFFLINE", "1")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

import re
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

from agenttune.agentic import Project, DictToolHarness, ReActStrategy, agentic_metrics
from agenttune.agentic.rollout_engines.transformers_engine import TransformersRolloutEngine

MODEL = "HuggingFaceTB/SmolLM2-360M-Instruct"

## Giving the assistant a simple tool it can choose to use

The assistant is told about one tool — a lookup — and is free to decide for itself, question by question, whether it needs to use it or can just answer directly.

In [2]:
REACT_SYS = (
    "You are a ReAct agent with ONE tool: lookup(q) — it looks up the answer to a factual "
    "question.\nTo use it, output EXACTLY: TOOL: lookup | ARGS: {\"q\": \"<question>\"}\n"
    "Once you know the answer, output EXACTLY: FINISH: <answer>\n"
    "Output ONLY one such line, nothing else."
)
FEWSHOT_U = "Question: what is the capital of Japan?"
FEWSHOT_A = 'TOOL: lookup | ARGS: {"q": "capital of Japan"}'
_TOOL_RE = re.compile(r'\{.*?"q"\s*:\s*"([^"]*)".*?\}', re.S)
_FINISH_RE = re.compile(r"FINISH:\s*(.+)")

FACTS = {"capital of france": "Paris"}


def lookup(q: str = "") -> str:
    """The real tool: a plain deterministic lookup — the model decides when to call it."""
    return FACTS.get(q.strip().lower(), f"no record for {q!r}")


def _generate(model, tok, messages, max_new_tokens):
    enc = tok.apply_chat_template(messages, add_generation_prompt=True,
                                  return_tensors="pt", return_dict=True).to(model.device)
    with torch.no_grad():
        out = model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False,
                             pad_token_id=tok.pad_token_id or tok.eos_token_id)
    return tok.decode(out[0, enc["input_ids"].shape[1]:], skip_special_tokens=True).strip()


def make_policy(model, tok):
    """Model-driven ReAct policy: the model decides whether to call `lookup`, every step."""
    def policy(state):
        if state.step == 0:
            raw = _generate(model, tok, [
                {"role": "system", "content": REACT_SYS},
                {"role": "user", "content": FEWSHOT_U}, {"role": "assistant", "content": FEWSHOT_A},
                {"role": "user", "content": "Question: " + state.task},
            ], max_new_tokens=32)
            m = _TOOL_RE.search(raw)
            if m:
                return {"name": "lookup", "arguments": {"q": m.group(1)}, "thought": raw}
            f = _FINISH_RE.search(raw)
            if f:
                return {"name": "finish", "arguments": {"answer": f.group(1).strip()}, "thought": raw}
            return {"name": "finish", "arguments": {"answer": raw[:40]}, "thought": raw}
        result = state.events[-1].payload.get("text", "")
        raw = _generate(model, tok, [
            {"role": "system", "content": REACT_SYS},
            {"role": "user", "content": "Question: " + state.task},
            {"role": "assistant", "content": FEWSHOT_A},
            {"role": "user", "content": f"lookup returned: {result}"},
        ], max_new_tokens=16)
        f = _FINISH_RE.search(raw)
        answer = f.group(1).strip() if f else result
        return {"name": "finish", "arguments": {"answer": answer}, "thought": raw}
    return policy

## Starting up the AI model

In [3]:
print(f"[gpu]     {torch.cuda.get_device_name(0)}  (cuda: {torch.cuda.is_available()})")
tok = AutoTokenizer.from_pretrained(MODEL)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token
model = AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype=torch.bfloat16).to("cuda")
print(f"[model]   {MODEL} loaded")

[gpu]     NVIDIA L40S  (cuda: True)


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

[model]   HuggingFaceTB/SmolLM2-360M-Instruct loaded


## Step 1 — Ask the assistant a question and watch it decide what to do

The assistant reads the question and decides on its own whether to use the lookup tool or answer right away.

In [4]:
harness = DictToolHarness({"lookup": lookup}, max_steps=4)
proj = Project(strategy=ReActStrategy(make_policy(model, tok), max_steps=4), harness=harness)

log = proj.infer("what is the capital of France?")
print(f"[build]   one episode -> {len(log)} events, tier={log.tier}")

[build]   one episode -> 9 events, tier=light


## Step 2 — Score how well it did

A set of automatic, rule-based checks grades the assistant's behavior — no human review needed.

In [5]:
metrics = agentic_metrics(log)
print(f"[eval]    programmatic metrics -> {metrics}")

[eval]    programmatic metrics -> {'tac': 0.0, 'ter': 0.5, 'arr': 0.0, 'scsr': 1.0, 'rad': 0.0, 'lcf': 0}


## Step 3 — Capture everything that happened, in detail

This is the part that makes future improvement possible: a detailed, structured recording of the assistant's reasoning is captured — not just the final answer — so it can be reused later.

In [6]:
tasks = ["what is 2+3?", "capital of France?"]
engine = TransformersRolloutEngine(model, tok)
logs = proj.collect_rollout(engine, tasks, tools=[], max_steps=2)
print(f"[collect] {len(logs)} rollouts, tiers={[l.tier for l in logs]}")

/workspace/agenttune/src/agenttune/agentic/project.py:140: UserWarning: No tools provided — running in single-turn mode.
  fn = create_rollout_fn(


[collect] 2 rollouts, tiers=['full', 'full']


## Step 4 — Package the recording into training material

The captured recordings are turned into ready-to-use training examples, in the exact shape a training process would consume to make the assistant better next time.

In [7]:
rows = proj.sft_dataset()
print(f"[train]   SFT dataset -> {len(rows)} rows, row0 keys={list(rows[0].keys())}")
print(f"[train]   row0 first message role={rows[0]['messages'][0]['role']!r}")

stages = [e.stage for e in proj.events()]
print(f"[stream]  lifecycle stages -> {stages}")

[train]   SFT dataset -> 2 rows, row0 keys=['messages', 'segment_weights', 'loss_mask']
[train]   row0 first message role='assistant'
[stream]  lifecycle stages -> ['infer', 'infer', 'collect_rollout', 'collect_rollout']
